# NEXUS-AI — EXP-003A/004 Kaggle training

Controlled training for NEXUS-HSR Tiny vs its parameter-matched Transformer baseline.

**How to use (Kaggle):**
1. Upload this notebook (File → Import Notebook) or paste it into a new Kaggle notebook.
2. Settings → Accelerator → **GPU T4 x1** (one GPU; do not assume pooled VRAM).
3. Settings → Internet → **On** (needed to clone the repository).
4. Run all cells top to bottom.

**Recorded configuration (fill in before running):**
- `REPO_COMMIT`: pin to a commit hash for strict reproducibility (`main` tracks latest).
- `SEEDS`, `EPOCHS`, `BATCH_SIZE`, `DATASET_SIZE`: experiment budget.
- `LIMIT_BATCHES`: leave `None` for full training; set a small number for a smoke run first.

Per project policy, hardware, software versions, precision, seed, and effective training
configuration are recorded into the outputs. Smoke tests on PC and GPU runs are kept
distinguishable via the saved summaries.

In [ ]:
# Experiment configuration — edit before Run All
REPO_URL = "https://github.com/RokhmadHD/NEXUS-AI.git"
REPO_COMMIT = "main"  # pin a full commit hash for strict reproducibility

SEEDS = [42, 43, 44]
EPOCHS = 10
BATCH_SIZE = 16
DATASET_SIZE = 10000        # records; splits are 80/10/10
DEVICE = "cuda"             # falls back to CPU in-cell if CUDA is unavailable
LIMIT_BATCHES = None        # e.g. 30 for a smoke run before the full sweep

OUTPUT_DIR = "/kaggle/working/runs"
ARTIFACTS_DIR = "/kaggle/working/artifacts"

In [ ]:
# Record hardware and software environment (required by AGENTS.md policy)
import json, os, subprocess, sys

def record(command):
    try:
        return subprocess.run(command, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as exc:
        return f"unavailable: {exc}"

import torch
environment = {
    "nvidia_smi": record(["nvidia-smi"]),
    "gpu_name": record(["bash", "-c", "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"]),
    "python": sys.version,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cpu_model": record(["bash", "-c", "grep -m1 'model name' /proc/cpuinfo"]),
}
if torch.cuda.is_available():
    environment["gpu_properties"] = {
        str(i): {
            "name": torch.cuda.get_device_name(i),
            "total_memory_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 2),
        }
        for i in range(torch.cuda.device_count())
    }
print(json.dumps(environment, indent=2))
assert DEVICE == "cuda" and torch.cuda.is_available(), (
    "GPU T4 runtime expected; enable the accelerator or set DEVICE='cpu'"
)

In [ ]:
# Clone the repository and install dependencies
# Note: the running kernel does not re-read .pth files, so we add the repo
# directory to sys.path directly instead of relying on the editable install.
import os, subprocess, sys

REPO = "nexus-repo"
subprocess.run(["rm", "-rf", REPO], check=True)
subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", REPO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", REPO], check=True)

REPO_HEAD = subprocess.run(
    ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
os.environ["REPO_HEAD"] = REPO_HEAD

if REPO not in sys.path:
    sys.path.insert(0, os.path.abspath(REPO))
print("using nexus-ai at commit:", REPO_HEAD)

import nexus_ai
from nexus_ai.data import VocabularyTokenizer
print("import ok; tokenizer API present:", hasattr(VocabularyTokenizer, "load"))


In [ ]:
# Generate the synthetic dataset, fit the tokenizer (train split only), validate
import subprocess, sys, json

REPO = "nexus-repo"
PY = sys.executable
DATASET_DIR = f"{REPO}/datasets/synthetic/v02_{DATASET_SIZE}"
TOKENIZER_PATH = f"{REPO}/datasets/processed/v02_{DATASET_SIZE}/tokenizer.json"

subprocess.run([PY, "-I", f"{REPO}/scripts/generate_dataset.py",
                "--output", DATASET_DIR, "--count", str(DATASET_SIZE), "--seed", "42"], check=True)
subprocess.run([PY, "-I", f"{REPO}/scripts/fit_tokenizer.py",
                "--train", f"{DATASET_DIR}/train.jsonl", "--output", TOKENIZER_PATH], check=True)
validation = subprocess.run(
    [PY, "-I", f"{REPO}/scripts/validate_dataset.py",
     f"{DATASET_DIR}/train.jsonl", f"{DATASET_DIR}/validation.jsonl", f"{DATASET_DIR}/test.jsonl"],
    capture_output=True, text=True, check=True,
)
tokenizer = VocabularyTokenizer.load(TOKENIZER_PATH)
print("tokenizer vocab_size:", tokenizer.vocab_size)
for record in json.loads(validation.stdout):
    print(record["path"], "records:", record["records"], "valid:", record["valid"], "errors:", len(record["errors"]))

In [ ]:
# Verify the parameter-matched budget with the fitted tokenizer
import subprocess, sys, json

result = subprocess.run(
    [sys.executable, "-I", f"{REPO}/scripts/compare_parameter_counts.py", TOKENIZER_PATH],
    capture_output=True, text=True, check=True,
)
report = json.loads(result.stdout)
print(json.dumps(report, indent=2))
assert report["parameter_matched"], "parameter budget mismatch"


## Training sweep

Both models share identical data order, tokenizer, objective, optimizer, batch size,
and update budget per seed. Only the architecture differs. Set `LIMIT_BATCHES`
(e.g. `30`) for a quick end-to-end smoke run before the full sweep.

In [ ]:
# Run the training sweep: {model} x {seed}
import subprocess, sys, time

os.makedirs(OUTPUT_DIR, exist_ok=True)
run_names = []
for seed in SEEDS:
    for model in ("nexus", "baseline"):
        name = f"{model}_s{seed}"
        run_dir = f"{OUTPUT_DIR}/{name}"
        command = [
            sys.executable, "-I", f"{REPO}/scripts/train.py",
            "--model", model,
            "--seed", str(seed),
            "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--tokenizer", TOKENIZER_PATH,
            "--train", f"{DATASET_DIR}/train.jsonl",
            "--validation", f"{DATASET_DIR}/validation.jsonl",
            "--device", DEVICE,
            "--output", run_dir,
        ]
        if LIMIT_BATCHES:
            command += ["--limit-batches", str(LIMIT_BATCHES)]
        started = time.time()
        subprocess.run(command, check=True)
        elapsed = time.time() - started
        print(f"[done] {name}: {elapsed:.1f}s")
        run_names.append(name)
print("completed runs:", run_names)

In [ ]:
# Evaluate every best checkpoint on the untouched test split
import subprocess, sys, json

results = []
for name in run_names:
    model = name.rsplit("_s", 1)[0]
    seed = name.rsplit("_s", 1)[1]
    proc = subprocess.run(
        [sys.executable, "-I", f"{REPO}/scripts/evaluate.py",
         "--model", model,
         "--checkpoint", f"{OUTPUT_DIR}/{name}/checkpoint_best.pt",
         "--tokenizer", TOKENIZER_PATH,
         "--data", f"{DATASET_DIR}/test.jsonl",
         "--device", DEVICE],
        capture_output=True, text=True, check=True,
    )
    payload = json.loads(proc.stdout)
    metrics = payload["metrics"]
    results.append({
        "run": name, "model": model, "seed": seed,
        "test_loss": round(metrics["loss"], 4),
        "test_accuracy": round(metrics["accuracy"], 4),
        "test_perplexity": round(metrics["perplexity"], 4),
    })

try:
    import pandas as pd
    results_frame = pd.DataFrame(results)
except ImportError:
    results_frame = results
print(results_frame)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
with open(f"{ARTIFACTS_DIR}/test_results.json", "w") as stream:
    json.dump(results, stream, indent=2)
if hasattr(results_frame, "to_csv"):
    results_frame.to_csv(f"{ARTIFACTS_DIR}/test_results.csv", index=False)

In [ ]:
# Aggregate per-model means and record the full run provenance
import json

summary = {}
for model in ("nexus", "baseline"):
    rows = [r for r in results if r["model"] == model]
    if rows:
        summary[model] = {
            "seeds": [r["seed"] for r in rows],
            "mean_test_loss": round(sum(r["test_loss"] for r in rows) / len(rows), 4),
            "mean_test_accuracy": round(sum(r["test_accuracy"] for r in rows) / len(rows), 4),
            "mean_test_perplexity": round(sum(r["test_perplexity"] for r in rows) / len(rows), 4),
        }

provenance = {
    "repo_commit": os.environ.get("REPO_HEAD", "unknown"),
    "config": {
        "seeds": SEEDS, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
        "dataset_size": DATASET_SIZE, "limit_batches": LIMIT_BATCHES,
        "device": DEVICE,
    },
    "environment": {k: v for k, v in environment.items() if k != "nvidia_smi"},
    "tokenizer_vocab_size": tokenizer.vocab_size,
    "per_model_means": summary,
    "note": "smoke tests do not establish learning or reasoning capability",
}
with open(f"{ARTIFACTS_DIR}/provenance.json", "w") as stream:
    json.dump(provenance, stream, indent=2)
print(json.dumps(provenance["per_model_means"], indent=2))

In [ ]:
# Package artifacts into a single downloadable zip
import shutil

archive = "/kaggle/working/nexus_ai_kaggle_results"
shutil.make_archive(archive, "zip", ARTIFACTS_DIR)
print("artifacts:")
for root, _, files in os.walk("/kaggle/working"):
    for name in files:
        path = os.path.join(root, name)
        print(f"  {path} ({os.path.getsize(path)} bytes)")